<a href="https://colab.research.google.com/github/fianbio/AI-driven-PLA2-antivenome/blob/main/02_feature_engineering_PLA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rdkit pubchempy pandas numpy tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/pla2'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

bioactivity_path = os.path.join(DATA_DIR, 'pla2_bioactivity_clean.csv')
approved_path = os.path.join(DATA_DIR, 'approved_drugs_chembl.csv')

print('Bioactivity file:', bioactivity_path, '- exists:', os.path.exists(bioactivity_path))
print('Approved drugs file:', approved_path, '- exists:', os.path.exists(approved_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bioactivity file: /content/drive/MyDrive/pla2/data/pla2_bioactivity_clean.csv - exists: True
Approved drugs file: /content/drive/MyDrive/pla2/data/approved_drugs_chembl.csv - exists: True


In [ ]:
import pandas as pd

bioactivity_df = pd.read_csv(bioactivity_path)
approved_df = pd.read_csv(approved_path)

print(f'Bioactivity dataset: {len(bioactivity_df)} baris')
print(f'Approved drugs dataset: {len(approved_df)} baris')

bioactivity_df.head()

Bioactivity dataset: 2030 baris
Approved drugs dataset: 3417 baris


,molecule_chembl_id,canonical_smiles,mean_standard_value,n_records,label
0,CHEMBL101015,CCCCCCCCCCCCCCCC(=O)OCC(COP(=O)(O)OCCO)NC(=O)C...,1765.0,2,0
1,CHEMBL101849,CCCCCCCCCCOc1cc(OCCCCCCCCCC)cc(N(CC(=O)O)CC(=O...,230.0,1,1
2,CHEMBL101890,CCCCCCCCCCCCCCCC(=O)NCCOP(=O)([O-])OCC[N+](C)(C)C,152000.0,2,0
3,CHEMBL101925,CC(C)C[C@H](COP(=O)([O-])OCC[N+](C)(C)C)NC(=O)...,160000.0,1,0
4,CHEMBL101972,CCCCCCCCCCCCCCCC(=O)N[C@@H](COP(=O)(O)OCCO)CC(C)C,1315.0,2,0


In [ ]:
bioactivity_df['label'].value_counts()

,count
label,
0,1328
1,702


In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
from tqdm import tqdm

RDLogger.DisableLog('rdApp.*')

MORGAN_RADIUS = 2
MORGAN_NBITS = 2048

def get_morgan_fp(smiles, radius=MORGAN_RADIUS, n_bits=MORGAN_NBITS):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

In [ ]:
import time
import base64
import pubchempy as pcp

PUBCHEM_FP_NUM_BITS = 881

def _decode_pubchem_fingerprint(fp_base64):
    raw_bytes = base64.b64decode(fp_base64)
    bits = np.unpackbits(np.frombuffer(raw_bytes, dtype=np.uint8))
    fp_bits = bits[32:32 + PUBCHEM_FP_NUM_BITS]
    return fp_bits.astype(np.int8)

def get_pubchem_fp(smiles, max_retries=5, sleep_sec=0.5):
     for attempt in range(max_retries):
        try:
            compounds = pcp.get_compounds(smiles, namespace='smiles')
            time.sleep(sleep_sec)  # jaga rate limit
            if not compounds:
                return None
            cid = compounds[0].cid
            if cid is None:
                return None
            props = pcp.get_properties(['Fingerprint2D'], cid, 'cid')
            time.sleep(sleep_sec)
            if not props or 'Fingerprint2D' not in props[0]:
                return None
            return _decode_pubchem_fingerprint(props[0]['Fingerprint2D'])
        except Exception as e:
            error_str = str(e)
            is_server_busy = 'ServerBusy' in error_str or '503' in error_str
            if attempt == max_retries - 1:
                print(f'Gagal mengambil PubChem FP untuk SMILES: {smiles[:50]}... | Error: {e}')
                return None
            if is_server_busy:
                # Backoff eksponensial lebih panjang khusus untuk server sibuk: 5s, 10s, 20s, 40s...
                wait_time = 5 * (2 ** attempt)
            else:
                wait_time = sleep_sec * (attempt + 2)
            time.sleep(wait_time)
     return None

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from rdkit.ML.Descriptors import MoleculeDescriptors

# Ambil semua nama descriptor yang tersedia di RDKit (~200 descriptor)
DESCRIPTOR_NAMES = [name for name, _ in Descriptors._descList]
_calculator = MoleculeDescriptors.MolecularDescriptorCalculator(DESCRIPTOR_NAMES)

def get_rdkit_descriptors(smiles):
    """
    Menghitung semua RDKit descriptors untuk satu SMILES.
    Mengembalikan numpy array (float32) atau None jika SMILES invalid.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        values = _calculator.CalcDescriptors(mol)
        arr = np.array(values, dtype=np.float32)
        # Ganti NaN/inf jadi 0 (kadang muncul untuk struktur tertentu)
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        return arr
    except Exception as e:
        print(f'Gagal menghitung descriptor untuk {smiles[:50]}... | Error: {e}')
        return None

In [ ]:
test_smiles = 'CC(=O)OC1=CC=CC=C1C(=O)O'

morgan_test = get_morgan_fp(test_smiles)
pubchem_test = get_pubchem_fp(test_smiles)
rdkit_test = get_rdkit_descriptors(test_smiles)

print('RDKit descriptors shape:', None if rdkit_test is None else rdkit_test.shape)
print('Morgan FP shape:', None if morgan_test is None else morgan_test.shape)
print('Morgan FP jumlah bit aktif:', None if morgan_test is None else morgan_test.sum())
print('PubChem FP shape:', None if pubchem_test is None else pubchem_test.shape)
print('PubChem FP jumlah bit aktif:', None if pubchem_test is None else pubchem_test.sum())

RDKit descriptors shape: (217,)
Morgan FP shape: (2048,)
Morgan FP jumlah bit aktif: 24
PubChem FP shape: (881,)
PubChem FP jumlah bit aktif: 115


In [ ]:
def generate_features_for_dataset(df, smiles_col, checkpoint_path, checkpoint_every=50):
    if os.path.exists(checkpoint_path):
        results = pd.read_pickle(checkpoint_path)
        print(f'Melanjutkan dari checkpoint: {len(results)} baris sudah diproses sebelumnya')
    else:
        results = pd.DataFrame(columns=['smiles', 'morgan_fp', 'pubchem_fp', 'rdkit_desc'])

    processed_smiles = set(results['smiles'].tolist())
    new_rows = []

    remaining = df[~df[smiles_col].isin(processed_smiles)]
    print(f'Sisa yang perlu diproses: {len(remaining)}')

    for i, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining), desc='Generate feature')):
        smiles = row[smiles_col]
        morgan_fp = get_morgan_fp(smiles)
        pubchem_fp = get_pubchem_fp(smiles)
        rdkit_desc = get_rdkit_descriptors(smiles)

        if morgan_fp is None or pubchem_fp is None or rdkit_desc is None:
            continue
        new_rows.append({'smiles': smiles, 'morgan_fp': morgan_fp, 'pubchem_fp': pubchem_fp, 'rdkit_desc': rdkit_desc})

        if (i + 1) % checkpoint_every == 0:
            checkpoint_df = pd.concat([results, pd.DataFrame(new_rows)], ignore_index=True)
            checkpoint_df.to_pickle(checkpoint_path)

    final_df = pd.concat([results, pd.DataFrame(new_rows)], ignore_index=True)
    final_df.to_pickle(checkpoint_path)
    return final_df

In [ ]:
bioactivity_checkpoint_path = os.path.join(DATA_DIR, 'bioactivity_fingerprints_checkpoint.pkl')

bioactivity_fp_df = generate_features_for_dataset(
    bioactivity_df,
    smiles_col='canonical_smiles',
    checkpoint_path=bioactivity_checkpoint_path,
)

print(f'Total baris dengan feature lengkap: {len(bioactivity_fp_df)}')

Melanjutkan dari checkpoint: 1963 baris sudah diproses sebelumnya
Sisa yang perlu diproses: 0


Generate feature: 0it [00:00, ?it/s]

Total baris dengan feature lengkap: 1963


In [ ]:
merged_bioactivity = bioactivity_df.merge(
    bioactivity_fp_df, left_on='canonical_smiles', right_on='smiles', how='left'
)

n_before = len(merged_bioactivity)
n_missing_morgan = merged_bioactivity['morgan_fp'].isna().sum()
n_missing_pubchem = merged_bioactivity['pubchem_fp'].isna().sum()
n_missing_descriptor = merged_bioactivity['rdkit_desc'].isna().sum()

print(f'Total baris sebelum filter: {n_before}')
print(f'Baris tanpa Morgan FP: {n_missing_morgan}')
print(f'Baris tanpa PubChem FP: {n_missing_pubchem}')
print(f'Baris tanpa RDKIT descriptor: {n_missing_descriptor}')

clean_bioactivity = merged_bioactivity.dropna(subset=['morgan_fp', 'pubchem_fp', 'rdkit_desc']).reset_index(drop=True)
print(f'Total baris setelah buang yang gagal: {len(clean_bioactivity)}')

Total baris sebelum filter: 2034
Baris tanpa Morgan FP: 0
Baris tanpa PubChem FP: 0
Baris tanpa RDKIT descriptor: 0
Total baris setelah buang yang gagal: 2034


In [ ]:
Xm_morgan = np.stack(clean_bioactivity['morgan_fp'].values)
Xm_pubchem = np.stack(clean_bioactivity['pubchem_fp'].values)
Xm_combined = np.concatenate([Xm_morgan, Xm_pubchem], axis=1)

y = clean_bioactivity['label'].values

print('Shape X_morgan:', Xm_morgan.shape)
print('Shape X_pubchem:', Xm_pubchem.shape)
print('Shape X_combined:', Xm_combined.shape)
print('Shape y:', y.shape)

Shape X_morgan: (2034, 2048)
Shape X_pubchem: (2034, 881)
Shape X_combined: (2034, 2929)
Shape y: (2034,)


In [ ]:
X_rdkit = np.stack(clean_bioactivity['rdkit_desc'].values)

X_morgan = np.concatenate([Xm_morgan, X_rdkit], axis=1)
X_pubchem = np.concatenate([Xm_pubchem, X_rdkit], axis=1)
X_combined = np.concatenate([Xm_combined, X_rdkit], axis=1)


print('Shape X_morgan:', X_morgan.shape)
print('Shape X_pubchem:', X_pubchem.shape)
print('Shape X_combined:', X_combined.shape)
print('Shape y:', y.shape)

Shape X_morgan: (2034, 2265)
Shape X_pubchem: (2034, 1098)
Shape X_combined: (2034, 3146)
Shape y: (2034,)


In [ ]:
features_path = os.path.join(DATA_DIR, 'training_features.npz')
np.savez_compressed(
    features_path,
    X_morgan=X_morgan,
    X_pubchem=X_pubchem,
    X_combined=X_combined,
    y=y,
)

metadata_path = os.path.join(DATA_DIR, 'training_metadata.csv')
clean_bioactivity[['molecule_chembl_id', 'canonical_smiles', 'mean_standard_value', 'label']].to_csv(
    metadata_path, index=False
)

print('Tersimpan di:', features_path)
print('Metadata tersimpan di:', metadata_path)

Tersimpan di: /content/drive/MyDrive/pla2/data/training_features.npz
Metadata tersimpan di: /content/drive/MyDrive/pla2/data/training_metadata.csv


In [ ]:
approved_checkpoint_path = os.path.join(DATA_DIR, 'approved_fingerprints_checkpoint.pkl')

approved_fp_df = generate_features_for_dataset(
    approved_df,
    smiles_col='canonical_smiles',
    checkpoint_path=approved_checkpoint_path,
)

print(f'Total baris dengan fingerprint: {len(approved_fp_df)}')

Melanjutkan dari checkpoint: 3411 baris sudah diproses sebelumnya
Sisa yang perlu diproses: 6


Generate feature:  17%|█▋        | 1/6 [00:10<00:50, 10.00s/it]

Gagal mengambil PubChem FP untuk SMILES: O=c1[n-]cnc2[nH]ncc12.[Na+]... | Error: PubChem HTTP Error 400 PUGREST.BadRequest: Unable to standardize the given structure - perhaps some special characters need to be escaped or data packed in a MIME form? (error: , status: 400, output: Caught ncbi::CException: Standardization failed, Output Log:, Record 1: Warning: Detected illegal valence for element "N": 2 sigma bonds, 1 pi bonds, -1 charge, Record 1: Warning: Compound failed verification during "standardize deposited compound", , PubChem Warning; class: Invalid Chemical Structure; label: Structure Standardization Issues; message: Detected illegal valence for element "N": 2 sigma bonds, 1 pi bonds, -1 charge, )


Generate feature:  67%|██████▋   | 4/6 [00:23<00:12,  6.13s/it]

Gagal mengambil PubChem FP untuk SMILES: Cl[223Ra]Cl... | Error: PubChem HTTP Error 400 PUGREST.BadRequest: Unable to standardize the given structure - perhaps some special characters need to be escaped or data packed in a MIME form? (error: , status: 400, output: Caught ncbi::CException: Standardization failed, Output Log:, Record 1: Warning: Detected illegal valence for element "Ra": 0 sigma bonds, 0 pi bonds, 2 charge, Record 1: Warning: Compound failed verification during "standardize deposited compound", , PubChem Warning; class: Invalid Chemical Structure; label: Structure Standardization Issues; message: Detected illegal valence for element "Ra": 0 sigma bonds, 0 pi bonds, 2 charge, )


Generate feature:  83%|████████▎ | 5/6 [00:33<00:07,  7.35s/it]

Gagal mengambil PubChem FP untuk SMILES: Cl[Ra]Cl... | Error: PubChem HTTP Error 400 PUGREST.BadRequest: Unable to standardize the given structure - perhaps some special characters need to be escaped or data packed in a MIME form? (error: , status: 400, output: Caught ncbi::CException: Standardization failed, Output Log:, Record 1: Warning: Detected illegal valence for element "Ra": 0 sigma bonds, 0 pi bonds, 2 charge, Record 1: Warning: Compound failed verification during "standardize deposited compound", , PubChem Warning; class: Invalid Chemical Structure; label: Structure Standardization Issues; message: Detected illegal valence for element "Ra": 0 sigma bonds, 0 pi bonds, 2 charge, )


Generate feature: 100%|██████████| 6/6 [00:43<00:00,  7.20s/it]

Gagal mengambil PubChem FP untuk SMILES: O=[As][As+](=O)[O-]... | Error: PubChem HTTP Error 400 PUGREST.BadRequest: Unable to standardize the given structure - perhaps some special characters need to be escaped or data packed in a MIME form? (error: , status: 400, output: Caught ncbi::CException: Standardization failed, Output Log:, Record 1: Warning: Detected illegal valence for element "As": 3 sigma bonds, 1 pi bonds, 1 charge, Record 1: Warning: Compound failed verification during "standardize deposited compound", , PubChem Warning; class: Invalid Chemical Structure; label: Structure Standardization Issues; message: Detected illegal valence for element "As": 3 sigma bonds, 1 pi bonds, 1 charge, )


Total baris dengan fingerprint: 3413


In [ ]:
merged_approved = approved_df.merge(
    approved_fp_df, left_on='canonical_smiles', right_on='smiles', how='left'
)

n_before = len(merged_approved)
clean_approved = merged_approved.dropna(subset=['morgan_fp', 'pubchem_fp', 'rdkit_desc']).reset_index(drop=True)
print(f'Total approved drugs sebelum filter: {n_before}')
print(f'Total approved drugs dengan fingerprint lengkap: {len(clean_approved)}')

Total approved drugs sebelum filter: 3417
Total approved drugs dengan fingerprint lengkap: 3413


In [ ]:
X_rdkit_app = np.stack(clean_approved['rdkit_desc'].values)
X_morgan_app = np.stack(clean_approved['morgan_fp'].values)
X_pubchem_app = np.stack(clean_approved['pubchem_fp'].values)
X_combined_app = np.concatenate([X_morgan_app, X_pubchem_app], axis=1)

print('Shape X_morgan_app:', X_morgan_app.shape)
print('Shape X_pubchem_app:', X_pubchem_app.shape)
print('Shape X_combined_app:', X_combined_app.shape)

Shape X_morgan_app: (3413, 2048)
Shape X_pubchem_app: (3413, 881)
Shape X_combined_app: (3413, 2929)


In [ ]:
X_morgan_approved = np.concatenate([X_morgan_app, X_rdkit_app], axis=1)
X_pubchem_approved = np.concatenate([X_pubchem_app, X_rdkit_app], axis=1)
X_combined_approved = np.concatenate([X_morgan_app, X_pubchem_app, X_rdkit_app], axis=1)

print(X_morgan_approved.shape)
print(X_pubchem_approved.shape)
print(X_combined_approved.shape)

(3413, 2265)
(3413, 1098)
(3413, 3146)


In [ ]:
print('Shape X_combined_approved:', X_combined_approved.shape)

approved_features_path = os.path.join(DATA_DIR, 'approved_drugs_features.npz')
np.savez_compressed(
    approved_features_path,
    X_morgan=X_morgan_approved,
    X_pubchem=X_pubchem_approved,
    X_combined=X_combined_approved,
)

approved_metadata_path = os.path.join(DATA_DIR, 'approved_drugs_metadata.csv')
clean_approved[['molecule_chembl_id', 'pref_name', 'canonical_smiles']].to_csv(
    approved_metadata_path, index=False
)

print('Tersimpan di:', approved_features_path)
print('Metadata tersimpan di:', approved_metadata_path)

Shape X_combined_approved: (3413, 3146)
Tersimpan di: /content/drive/MyDrive/pla2/data/approved_drugs_features.npz
Metadata tersimpan di: /content/drive/MyDrive/pla2/data/approved_drugs_metadata.csv
